### Initialisation

In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Blues_d")

In [12]:
# Load the training data
train_df = pd.read_csv('../data/application_train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Default rate: {train_df['TARGET'].mean():.2%}")
print(f"Missing value %: {(train_df.isnull().sum().sum() / train_df.size) * 100:.1f}%")

Training data shape: (307511, 122)
Default rate: 8.07%
Missing value %: 24.4%


## 1

### Feature Engineering

In [ ]:
# core features
train_df['CREDIT_INCOME_RATIO'] = train_df['AMT_CREDIT'] / train_df['AMT_INCOME_TOTAL'].replace(0, np.nan)
train_df['ANNUITY_INCOME_RATIO'] = train_df['AMT_ANNUITY'] / train_df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# days to years
train_df['DAYS_BIRTH_YEARS'] = -train_df['DAYS_BIRTH'] / 365
train_df['DAYS_EMPLOYED_YEARS'] = train_df['DAYS_EMPLOYED'].apply(lambda x: -x/365 if x < 0 else np.nan)

# feature: ratio of employment to age
train_df['EMPLOYMENT_TO_AGE_RATIO'] = train_df['DAYS_EMPLOYED_YEARS'] / train_df['DAYS_BIRTH_YEARS']

# feature: family size feature
train_df['FAMILY_SIZE'] = train_df['CNT_FAM_MEMBERS'].fillna(0) + 1

# feature: thin file flag (if all 3 external sources missing -> flag as thin file)
train_df['THIN_FILE_FLAG'] = train_df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isnull().all(axis=1).astype(int)

print(f"Shape after new features added: {train_df.shape}")

Shape after new features added: (307511, 129)


### Segment Analysis

In [ ]:
# Segment 1: High debt burden
high_debt = train_df['CREDIT_INCOME_RATIO'] > 3.5
default_high_debt = train_df[high_debt]['TARGET'].mean()
default_low_debt = train_df[~high_debt]['TARGET'].mean()
print(f"\n  High debt burden (>3.5x): {default_high_debt:.2%} default rate")
print(f"  Low debt burden: {default_low_debt:.2%} default rate")
print(f"  Risk multiplier: {default_high_debt/default_low_debt:.1f}x")

# Segment 2: Thin-file applicants
thin_default = train_df[train_df['THIN_FILE_FLAG'] == 1]['TARGET'].mean()
thick_default = train_df[train_df['THIN_FILE_FLAG'] == 0]['TARGET'].mean()
print(f"\n  Thin-file applicants: {thin_default:.2%} default rate")
print(f"  With bureau history: {thick_default:.2%} default rate")
print(f"  Risk multiplier: {thin_default/thick_default:.1f}x")

# Segment 3: External score analysis 
if 'EXT_SOURCE_2' in train_df.columns:
    low_score = train_df['EXT_SOURCE_2'] < 0.3
    high_score = train_df['EXT_SOURCE_2'] > 0.6
    if low_score.any() and high_score.any():
        print(f"\n  Low EXT_SOURCE_2 (<0.3): {train_df[low_score]['TARGET'].mean():.2%} default")
        print(f"  High EXT_SOURCE_2 (>0.6): {train_df[high_score]['TARGET'].mean():.2%} default")

# Segment 4: Employment stability
short_emp = train_df['DAYS_EMPLOYED_YEARS'] < 1
long_emp = train_df['DAYS_EMPLOYED_YEARS'] > 5
if short_emp.any() and long_emp.any():
    print(f"\n  Employed <1 year: {train_df[short_emp]['TARGET'].mean():.2%} default")
    print(f"  Employed >5 years: {train_df[long_emp]['TARGET'].mean():.2%} default")


  High debt burden (>3.5x): 7.95% default rate
  Low debt burden: 8.18% default rate
  Risk multiplier: 1.0x

  Thin-file applicants: 8.14% default rate
  With bureau history: 8.07% default rate
  Risk multiplier: 1.0x

  Low EXT_SOURCE_2 (<0.3): 15.88% default
  High EXT_SOURCE_2 (>0.6): 4.56% default

  Employed <1 year: 10.98% default
  Employed >5 years: 6.41% default


### Modelling Preparation

In [49]:
# Select features (exclude target and identifiers)
exclude_cols = ['TARGET', 'SK_ID_CURR']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

# Handle missing values - simple median imputation for now
X = train_df[feature_cols].copy()
y = train_df['TARGET'].copy()

# Convert object types
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

# Median impute numeric
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Class balance: {y_train.mean():.2%} default in training")

Training set: (246008, 127)
Validation set: (61503, 127)
Class balance: 8.07% default in training


### Baseline Model

In [50]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Simple Random Forest (faster than XGBoost for initial test)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train.select_dtypes(include=[np.number]), y_train)

# Predict
y_pred_proba = rf.predict_proba(X_val.select_dtypes(include=[np.number]))[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)
print(f"Baseline Random Forest ROC-AUC: {auc:.4f}")

Baseline Random Forest ROC-AUC: 0.7010


In [55]:
# Create summary dataframe for dashboard
segment_summary = pd.DataFrame({
    'Segment': ['High Debt Burden (>3.5x)', 'Low Debt Burden (≤3.5x)', 
                'Thin-File Applicants', 'With Bureau History',
                'Employed <1 Year', 'Employed >5 Years'],
    'Default_Rate': [default_high_debt, default_low_debt, 
                     thin_default, thick_default,
                     train_df[short_emp]['TARGET'].mean() if short_emp.any() else 0,
                     train_df[long_emp]['TARGET'].mean() if long_emp.any() else 0],
    'Sample_Size': [high_debt.sum(), (~high_debt).sum(),
                    train_df[train_df['THIN_FILE_FLAG']==1].shape[0],
                    train_df[train_df['THIN_FILE_FLAG']==0].shape[0],
                    short_emp.sum() if short_emp.any() else 0,
                    long_emp.sum() if long_emp.any() else 0]
})

segment_summary.to_csv('../outputs/segment_analysis.csv', index=False)
print("Saved segment_analysis.csv for dashboard")

# Save model
import joblib
joblib.dump(rf, '../outputs/baseline_model.pkl')
print("Saved baseline_model.pkl")

print("\n" + "=" * 60)
print("EDA + Feature Engineering complete!")
print("Next: Run modelling notebook to improve ROC-AUC")
print("=" * 60)

# Display key metrics for your consulting summary
print("\nKEY METRICS SUMMARY:")
print("-" * 40)
print(f"ROC-AUC (baseline): {auc:.3f}")
print(f"High-debt risk multiplier: {default_high_debt/default_low_debt:.1f}x")
print(f"Thin-file risk multiplier: {thin_default/thick_default:.1f}x")

Saved segment_analysis.csv for dashboard
Saved baseline_model.pkl

EDA + Feature Engineering complete!
Next: Run modelling notebook to improve ROC-AUC

KEY METRICS SUMMARY:
----------------------------------------
ROC-AUC (baseline): 0.701
High-debt risk multiplier: 1.0x
Thin-file risk multiplier: 1.0x


### Diagnostic

In [ ]:
# DIAGNOSTIC: Check if features were created correctly
print("\n" + "=" * 60)
print("DIAGNOSTIC: Checking Feature Creation")
print("=" * 60)

# Check 1: Did CREDIT_INCOME_RATIO get created?
if 'CREDIT_INCOME_RATIO' in train_df.columns:
    valid_ratios = train_df['CREDIT_INCOME_RATIO'].dropna()
    print(f"\n✓ CREDIT_INCOME_RATIO created")
    print(f"  - Range: {valid_ratios.min():.2f} to {valid_ratios.max():.2f}")
    print(f"  - Median: {valid_ratios.median():.2f}")
    print(f"  - % with ratio > 3.5: {(valid_ratios > 3.5).mean()*100:.1f}%")
else:
    print("\n✗ CREDIT_INCOME_RATIO NOT found — check your formula")

# Check 2: Did THIN_FILE_FLAG get created?
if 'THIN_FILE_FLAG' in train_df.columns:
    print(f"\n✓ THIN_FILE_FLAG created")
    print(f"  - % thin-file: {train_df['THIN_FILE_FLAG'].mean()*100:.1f}%")
else:
    print("\n✗ THIN_FILE_FLAG NOT found — check your logic")



DIAGNOSTIC: Checking Feature Creation

✓ CREDIT_INCOME_RATIO created
  - Range: 0.00 to 84.74
  - Median: 3.27
  - % with ratio > 3.5: 45.7%

✓ THIN_FILE_FLAG created
  - % thin-file: 0.1%


In [61]:
# Check 3: Look at the actual default rates by segment
print("\n" + "-" * 40)
print("ACTUAL DEFAULT RATES BY SEGMENT:")
print("-" * 40)

# Debt ratio segments
if 'CREDIT_INCOME_RATIO' in train_df.columns:
    # Create groups
    low_debt = train_df['CREDIT_INCOME_RATIO'] <= 2
    med_debt = (train_df['CREDIT_INCOME_RATIO'] > 2) & (train_df['CREDIT_INCOME_RATIO'] <= 3.5)
    high_debt = train_df['CREDIT_INCOME_RATIO'] > 3.5
    
    print(f"\nDebt-to-Income (DTI) Analysis:")
    print(f"  Low DTI (≤2x):     {train_df[low_debt]['TARGET'].mean():.2%} default — {low_debt.sum():,} applicants")
    print(f"  Medium DTI (2-3.5x): {train_df[med_debt]['TARGET'].mean():.2%} default — {med_debt.sum():,} applicants")
    print(f"  High DTI (>3.5x):  {train_df[high_debt]['TARGET'].mean():.2%} default — {high_debt.sum():,} applicants")

# Check what's in EXT_SOURCE_2
if 'EXT_SOURCE_2' in train_df.columns:
    print(f"\nExternal Credit Score (EXT_SOURCE_2):")
    has_score = train_df['EXT_SOURCE_2'].notna()
    print(f"  % with score: {has_score.mean()*100:.1f}%")
    if has_score.any():
        low_score = train_df['EXT_SOURCE_2'] < 0.3
        high_score = train_df['EXT_SOURCE_2'] > 0.6
        print(f"  Low score (<0.3): {train_df[low_score]['TARGET'].mean():.2%} default")
        print(f"  High score (>0.6): {train_df[high_score]['TARGET'].mean():.2%} default")

# Check employment data
if 'DAYS_EMPLOYED_YEARS' in train_df.columns:
    print(f"\nEmployment Tenure:")
    short_emp = train_df['DAYS_EMPLOYED_YEARS'] < 1
    long_emp = train_df['DAYS_EMPLOYED_YEARS'] > 5
    print(f"  Employed <1 year:  {train_df[short_emp]['TARGET'].mean():.2%} default")
    print(f"  Employed >5 years: {train_df[long_emp]['TARGET'].mean():.2%} default")


----------------------------------------
ACTUAL DEFAULT RATES BY SEGMENT:
----------------------------------------

Debt-to-Income (DTI) Analysis:
  Low DTI (≤2x):     7.43% default — 76,338 applicants
  Medium DTI (2-3.5x): 8.81% default — 90,638 applicants
  High DTI (>3.5x):  7.95% default — 140,535 applicants

External Credit Score (EXT_SOURCE_2):
  % with score: 99.8%
  Low score (<0.3): 15.88% default
  High score (>0.6): 4.56% default

Employment Tenure:
  Employed <1 year:  10.98% default
  Employed >5 years: 6.41% default


## 2

### Feature Engineering

In [62]:
df = train_df.copy()

# Select features that matter most (based on diagnostic)
key_features = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1',  # External scores
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',     # Debt ratios
    'DAYS_EMPLOYED_YEARS', 'DAYS_BIRTH_YEARS',         # Tenure & age
    'AMT_CREDIT', 'AMT_INCOME_TOTAL',                  # Loan & income size
    'CNT_FAM_MEMBERS',                                 # Family size
    'EMPLOYMENT_TO_AGE_RATIO',                         # Stability metric
]

# Add any other numeric columns that exist
numeric_cols = df.select_dtypes(include=[np.number]).columns
additional_cols = [c for c in numeric_cols if c not in key_features + ['TARGET', 'SK_ID_CURR']]
key_features.extend(additional_cols[:30])  # Limit to 30 features

# Create feature matrix
X = df[key_features].copy()
y = df['TARGET'].copy()

print(f"Using {X.shape[1]} features")

# Handle missing values
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0)


Using 41 features


### Train-Test Split

In [63]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Default rate in training: {y_train.mean():.2%}")

Training: 246,008 samples
Validation: 61,503 samples
Default rate in training: 8.07%


### Train Optimised Model

In [64]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight='balanced',  # Critical for imbalanced data
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_proba = rf_model.predict_proba(X_val)[:, 1]
y_pred_class = (y_pred_proba > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_val, y_pred_proba)
print(f"\nROC-AUC: {auc:.4f}")


ROC-AUC: 0.7367


### Optimal Threshold for Business Use

In [65]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

# Find threshold that captures 80% of defaults (high recall)
target_recall = 0.80
closest_idx = (np.abs(recalls - target_recall)).argmin()
optimal_threshold = thresholds[closest_idx] if closest_idx < len(thresholds) else 0.5

print(f"Threshold to capture 80% of defaults: {optimal_threshold:.3f}")

# Apply optimal threshold
y_pred_optimal = (y_pred_proba > optimal_threshold).astype(int)

Threshold to capture 80% of defaults: 0.409


### Segment Analysis with Model Predictions

In [66]:
# Create results dataframe
results = X_val.copy()
results['actual_default'] = y_val
results['pred_prob'] = y_pred_proba
results['predicted_risky'] = y_pred_optimal

# Segment 1: By credit score
if 'EXT_SOURCE_2' in results.columns:
    low_score = results['EXT_SOURCE_2'] < 0.3
    high_score = results['EXT_SOURCE_2'] > 0.6
    
    print(f"\nEXT_SOURCE_2 segments:")
    print(f"  Low score (<0.3): Model predicted {results[low_score]['predicted_risky'].mean():.1%} as risky")
    print(f"  High score (>0.6): Model predicted {results[high_score]['predicted_risky'].mean():.1%} as risky")

# Segment 2: By debt ratio
if 'CREDIT_INCOME_RATIO' in results.columns:
    high_debt = results['CREDIT_INCOME_RATIO'] > 3.5
    print(f"\nHigh debt (>3.5x): Model flagged {results[high_debt]['predicted_risky'].mean():.1%} as risky")


EXT_SOURCE_2 segments:
  Low score (<0.3): Model predicted 84.9% as risky
  High score (>0.6): Model predicted 26.8% as risky

High debt (>3.5x): Model flagged 48.3% as risky


### Feature Importance

In [67]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
for i, row in feature_importance.head(10).iterrows():
    print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")


Top 10 Most Important Features:
  1. EXT_SOURCE_2: 0.2312
  2. EXT_SOURCE_3: 0.2302
  3. EXT_SOURCE_1: 0.0962
  6. DAYS_EMPLOYED_YEARS: 0.0434
  7. DAYS_BIRTH_YEARS: 0.0406
  11. EMPLOYMENT_TO_AGE_RATIO: 0.0344
  16. DAYS_BIRTH: 0.0332
  14. AMT_GOODS_PRICE: 0.0301
  8. AMT_CREDIT: 0.0250
  17. DAYS_EMPLOYED: 0.0249


### Save Model

In [70]:
import joblib
joblib.dump(rf_model, '../outputs/credit_risk_model.pkl')
print("✓ Model saved to outputs/credit_risk_model.pkl")

# Save feature importance for dashboard
feature_importance.to_csv('../outputs/feature_importance.csv', index=False)
print("✓ Feature importance saved")

# Save segment analysis for dashboard
segment_data = pd.DataFrame({
    'Segment': ['Low Credit Score (<0.3)', 'High Credit Score (>0.6)',
                'Short Employment (<1 year)', 'Long Employment (>5 years)',
                'Medium DTI (2-3.5x)', 'Low DTI (≤2x)', 'High DTI (>3.5x)'],
    'Default_Rate': [15.88, 4.56, 10.98, 6.41, 8.81, 7.43, 7.95],
    'Sample_Size': [
        (train_df['EXT_SOURCE_2'] < 0.3).sum(),
        (train_df['EXT_SOURCE_2'] > 0.6).sum(),
        (train_df['DAYS_EMPLOYED_YEARS'] < 1).sum(),
        (train_df['DAYS_EMPLOYED_YEARS'] > 5).sum(),
        90638, 76338, 140535
    ]
})
segment_data.to_csv('../outputs/segment_analysis.csv', index=False)
print("✓ Segment analysis saved")

✓ Model saved to outputs/credit_risk_model.pkl
✓ Feature importance saved
✓ Segment analysis saved


## 3

### Final Summary

In [88]:
print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                        │
├─────────────────────────────────────────────────────────────┤
│  ROC-AUC (Random Forest):     {auc:.3f}                         │
│  Baseline comparison:         0.701 → {auc:.3f}                 │
│  Improvement:                 +{(auc - 0.701)*100:.1f} percentage pts           │
├─────────────────────────────────────────────────────────────┤
│                    RISK SEGMENTS                            │
├─────────────────────────────────────────────────────────────┤
│  Low EXT_SOURCE_2 (<0.3):      15.88% default (3.5x risk)   │
│  High EXT_SOURCE_2 (>0.6):      4.56% default (baseline)    │
│  Employed <1 year:             10.98% default (1.7x risk)   │
│  Employed >5 years:             6.41% default               │
│  Medium DTI (2-3.5x):           8.81% default (peak risk)   │
├─────────────────────────────────────────────────────────────┤
│                    TOP FEATURES                             │
├─────────────────────────────────────────────────────────────┤
│  #1: {feature_importance.iloc[0]['feature'][:25]}                                           │
│  #2: {feature_importance.iloc[1]['feature'][:25] if len(feature_importance) > 1 else 'N/A'}                                           │
│  #3: {feature_importance.iloc[2]['feature'][:25] if len(feature_importance) > 2 else 'N/A'}                                           │
└─────────────────────────────────────────────────────────────┘
""")


┌─────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                        │
├─────────────────────────────────────────────────────────────┤
│  ROC-AUC (Random Forest):     0.737                         │
│  Baseline comparison:         0.701 → 0.737                 │
│  Improvement:                 +3.6 percentage pts           │
├─────────────────────────────────────────────────────────────┤
│                    RISK SEGMENTS                            │
├─────────────────────────────────────────────────────────────┤
│  Low EXT_SOURCE_2 (<0.3):      15.88% default (3.5x risk)   │
│  High EXT_SOURCE_2 (>0.6):      4.56% default (baseline)    │
│  Employed <1 year:             10.98% default (1.7x risk)   │
│  Employed >5 years:             6.41% default               │
│  Medium DTI (2-3.5x):           8.81% default (peak risk)   │
├─────────────────────────────────────────────────────────────┤
│                    TOP FEATURES      